# FinCouncil AI: Multi-Agent Financial Committee & Literacy Coach

Welcome to the **FinCouncil AI** Jupyter Notebook! This notebook implements the core multi-agent deliberation, risk-quantification, and financial education framework using the new **Google GenAI SDK** and **Gemini 2.5**.

### 🏆 Capstone Project Track: Agents for Business
* **Live Dashboard**: [https://fincouncil-ai.vercel.app](https://fincouncil-ai.vercel.app)
* **GitHub Repository**: [https://github.com/Siva-Subramaniam-DS/fincouncil-ai](https://github.com/Siva-Subramaniam-DS/fincouncil-ai)

### 🤖 The 6 Specialized Agents
1. **Executive Agent (Chief Investment Strategist)**: Oversees the deliberation, directs the sub-agents, and produces the final synthesis.
2. **Market Agent (Quantitative Analyst)**: Evaluates technicals, valuations, and fundamentals.
3. **News Agent (Sentiment Intelligence)**: Scans public sentiment, earnings calls transcripts, and media narratives.
4. **Risk Agent (Risk Management Officer)**: Flags systemic downsides, stop-losses, and downside exposure.
5. **Committee Agent (Deliberation Facilitator)**: Moderates debate, acts as the devil's advocate, and resolves consensus.
6. **Education Agent (Financial Literacy Coach)**: Automatically scans transcripts, extracts complex jargon, and translates it into simplified concepts.

## 🛠️ Step 1: Install Dependencies
First, we will install the new official Google GenAI Python library (`google-genai`) and `pydantic` for structured response validation.

In [ ]:
!pip install -q google-genai pydantic

## 🔑 Step 2: Configure Gemini API Key
We load the API key from Kaggle Secrets (or fallback to environment variables if running locally).

In [ ]:
import os
from google import genai
from google.genai import types

# Attempt to fetch from Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = api_key
except Exception:
    # Fallback to local env variable
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    print("⚠️ GEMINI_API_KEY not found. Please set it in Kaggle Secrets or env variables.")
else:
    print("✅ Gemini API client successfully configured!")

client = genai.Client()

## 📋 Step 3: Define Structured Output Pydantic Schemas
To ensure reliability and enable clean visualization in our frontend dashboard, we force Gemini to output exact JSON matching our schemas using Pydantic.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class AgentAnalysis(BaseModel):
    agent_name: str = Field(description="The role name of the agent (e.g., Quant Analyst)")
    verdict: str = Field(description="Summary of the sub-agent's findings")
    key_metrics: List[str] = Field(description="Valuations or signals analyzed")
    bull_points: List[str] = Field(description="Positive catalysts identified")
    bear_points: List[str] = Field(description="Downside catalysts identified")
    score: float = Field(description="Sentiment score from 0 (Highly Bearish) to 100 (Highly Bullish)")

class DebateTurn(BaseModel):
    agent_name: str = Field(description="The agent speaking")
    statement: str = Field(description="The argument or response in the debate")

class DebateModeration(BaseModel):
    summary: str = Field(description="Brief summary of the debate consensus")
    turns: List[DebateTurn] = Field(description="The chronological dialog turns")

class FinalSynthesis(BaseModel):
    consensus_verdict: str = Field(description="E.g., Moderate Buy, Hold, Strong Sell")
    consensus_score: float = Field(description="Weighted confidence score from 0 to 100")
    position_sizing_limit: str = Field(description="Position size limit (e.g., 4-6% max)")
    stop_loss_parameters: str = Field(description="Suggested stop-loss triggers")
    summary_thesis: str = Field(description="The final consolidated thesis from the Chief Strategist")

class JargonNote(BaseModel):
    term: str = Field(description="The financial term unpacked (e.g. Dollar-Cost Averaging)")
    definition: str = Field(description="A plain-language translation for retail investors")
    actionable_lesson: str = Field(description="How to apply this concept practically")

class EducationOutput(BaseModel):
    notes: List[JargonNote]

## 🤖 Step 4: Define Agent System Instructions
Here we create the personas and prompts for each of our 6 specialists.

In [ ]:
PROMPTS = {
    "executive_route": """
You are the Executive Agent (Chief Investment Strategist) of the FinCouncil. 
Your job is to analyze the user's financial query and output a routing roadmap explaining what factors the Quantitative Analyst (Market), Sentiment Intelligence (News), and Risk Officer must scan.
Keep it brief, structuring it as direct objectives.
""",
    
    "market_agent": """
You are the Market Agent (Quantitative Analyst). You evaluate technical indicators, valuation metrics (P/E ratios, forward multipliers), and growth drivers. 
You must focus on numeric data, targets, and trends.
""",
    
    "news_agent": """
You are the News Agent (Sentiment Intelligence). You analyze qualitative news, press releases, public sentiment, and earnings call transcripts. 
Gravel with brand strength, media narratives, and industry perception.
""",
    
    "risk_agent": """
You are the Risk Agent (Risk Management Officer). Your mandate is identifying downside exposures, macro threats, regulatory hurdles, supply chain delays, and interest rate exposure. 
Be highly critical and defensive.
""",
    
    "committee_agent": """
You are the Committee Agent (Deliberation Facilitator). You moderate a structured debate between the sub-agents. 
Challenge their assertions. Act as devil's advocate. Force the Quant and Risk agents to respond to each other's points until a middle-ground is found.
""",
    
    "executive_synthesis": """
You are the Executive Agent. Review the quantitative reviews, sentiment assessments, risk warnings, and the moderated debate log.
Synthesize the final consensus recommendation. Calculate the recommended position sizing limits (e.g. 2-3%, 5-8% max) and stop-loss parameters.
""",
    
    "education_agent": """
You are the Education Agent (Financial Literacy Coach). Scan the entire deliberation logs and reports generated.
Identify any complex financial jargon, technical abbreviations, or math terms (e.g. Forward P/E, stop-loss, options, dollar-cost averaging).
Produce a plain-language glossary explaining these terms to a first-time investor.
"""
}

## ⚙️ Step 5: Implement Multi-Agent Deliberation Orchestrator
We build the pipeline function that routes queries through all 6 agents sequentially, capturing each output.

In [ ]:
import json

def run_fincouncil_deliberation(query: str):
    print(f"🔮 Initial Query: {query}\n")
    
    # 1. Executive Agent routes the query
    print("🔄 1. Executive Agent creating routing roadmap...")
    route_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Query: {query}",
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["executive_route"]
        )
    )
    route_plan = route_resp.text
    print(f"-> Routing Plan: {route_plan.strip()}\n")

    # 2. Parallel scans by sub-agents using Structured Outputs
    print("🔄 2. Market, News, and Risk agents executing parallel analysis...")
    
    # Market Agent
    market_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Query: {query}\nRoute Directive: {route_plan}",
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["market_agent"],
            response_mime_type="application/json",
            response_schema=AgentAnalysis
        )
    )
    market_analysis = json.loads(market_resp.text)
    print("✅ Market analysis report complete.")
    
    # News Agent
    news_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Query: {query}\nRoute Directive: {route_plan}",
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["news_agent"],
            response_mime_type="application/json",
            response_schema=AgentAnalysis
        )
    )
    news_analysis = json.loads(news_resp.text)
    print("✅ News analysis report complete.")

    # Risk Agent
    risk_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Query: {query}\nRoute Directive: {route_plan}",
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["risk_agent"],
            response_mime_type="application/json",
            response_schema=AgentAnalysis
        )
    )
    risk_analysis = json.loads(risk_resp.text)
    print("✅ Risk analysis report complete.\n")

    # 3. Moderated debate between agents
    print("🔄 3. Committee Agent initiating structured debate...")
    debate_context = f"""
Market Analyst Report: {json.dumps(market_analysis)}
News Analyst Report: {json.dumps(news_analysis)}
Risk Analyst Report: {json.dumps(risk_analysis)}
"""
    debate_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Analyze this context and moderate a debate: {debate_context}",
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["committee_agent"],
            response_mime_type="application/json",
            response_schema=DebateModeration
        )
    )
    debate_log = json.loads(debate_resp.text)
    print("✅ Committee debate transcripts simulated.\n")

    # 4. Executive Agent synthesis (Gemini Pro for maximum cognitive capability)
    print("🔄 4. Executive Agent synthesizing final thesis and position sizing...")
    synthesis_context = f"""
Original Query: {query}
Reports Context: {debate_context}
Debate transcripts: {json.dumps(debate_log)}
"""
    synthesis_resp = client.models.generate_content(
        model="gemini-2.5-pro", # Pro works best for final deliberation/policy synthesis
        contents=synthesis_context,
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["executive_synthesis"],
            response_mime_type="application/json",
            response_schema=FinalSynthesis
        )
    )
    final_thesis = json.loads(synthesis_resp.text)
    print("✅ Final consensus thesis generated.\n")

    # 5. Education Agent extracts financial terms and creates a glossary
    print("🔄 5. Education Agent generating financial literacy glossary...")
    education_context = f"""
Complete deliberation record: {synthesis_context}
Final Thesis: {json.dumps(final_thesis)}
"""
    education_resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=education_context,
        config=types.GenerateContentConfig(
            system_instruction=PROMPTS["education_agent"],
            response_mime_type="application/json",
            response_schema=EducationOutput
        )
    )
    glossary = json.loads(education_resp.text)
    print("✅ Educational glossary created.\n")

    return {
        "route_plan": route_plan,
        "market_analysis": market_analysis,
        "news_analysis": news_analysis,
        "risk_analysis": risk_analysis,
        "debate_log": debate_log,
        "final_thesis": final_thesis,
        "glossary": glossary
    }

## 📈 Step 6: Render and Visualize Reports
We create a markdown renderer to display the outputs of our multi-agent committee in a clean, human-readable layout.

In [ ]:
from IPython.display import display, Markdown

def display_fincouncil_report(results):
    markdown_content = f"""
# 🛡️ FinCouncil AI Consensus Report
### Verdict: **{results['final_thesis']['consensus_verdict']}** (Confidence: **{results['final_thesis']['consensus_score']}%**)

---

## 📊 Executive Recommendation
**Strategic Thesis:**  
{results['final_thesis']['summary_thesis']}

**Portfolio Rules:**
* 📦 **Max Position Sizing:** `{results['final_thesis']['position_sizing_limit']}`
* 🛑 **Stop-Loss Strategy:** `{results['final_thesis']['stop_loss_parameters']}`

---

## 💬 Moderated Deliberation Feed
**Facilitator Summary:** *{results['debate_log']['summary']}*

"""
    
    for turn in results['debate_log']['turns']:
        markdown_content += f"* **{turn['agent_name']}**: \"{turn['statement'].strip()}\"\n"
        
    markdown_content += """

---

## 🔍 Sub-Agent Assessment Profiles
| Agent | Sentiment | Analyzed Metrics | Bull Case | Bear Case |
|---|---|---|---|---|
"""
    
    for agent in [results['market_analysis'], results['news_analysis'], results['risk_analysis']]:
        metrics_str = ", ".join(agent['key_metrics'])
        bull_str = "<br>".join([f"• {pt}" for pt in agent['bull_points']])
        bear_str = "<br>".join([f"• {pt}" for pt in agent['bear_points']])
        markdown_content += f"| **{agent['agent_name']}** | {agent['score']}% | {metrics_str} | {bull_str} | {bear_str} |\n"
        
    markdown_content += """

---

## 🎓 Educational Glossary (Literacy Coach)
"""
    
    for item in results['glossary']['notes']:
        markdown_content += f"""### 💡 **{item['term']}**
* **Definition:** {item['definition']}
* **Actionable Lesson:** *{item['actionable_lesson']}*\n\n"""

    display(Markdown(markdown_content))

## 🚀 Step 7: Run a Sample Deliberation
Let's ask the committee whether to invest in **NVIDIA (NVDA)** at its current prices.

In [ ]:
query = "Should I buy NVDA at current prices, or is it in a bubble?"
deliberation_results = run_fincouncil_deliberation(query)

In [ ]:
# Render the gorgeous final Markdown report
display_fincouncil_report(deliberation_results)